In [22]:
import random
import math
import pandas as pd
import numpy as np
import time
import copy



from preferences import prefs
from user import user_prefs, experiments

startPoint = [59.927085, 30.317504]
endPoint = [59.935408, 30.327108]

In [23]:
df = pd.read_csv('data/places.csv')

In [24]:
def point_interest_score(point_row):
    score = 0
    for category, tags in prefs.items():
        weight = user_prefs[category]
        for tag in tags:
            if tag in point_row.index and point_row[tag]:
                score += weight

    return score


# Бонус за разнообразие маршрута
def diversity_bonus(route, df):
    """
    Поощряем разнообразие категорий.
    """

    used_categories = set()

    for idx in route:

        point = df.iloc[idx]

        for category_name, tags in tag_groups.items():

            for tag in tags:

                if tag in point and point[tag]:
                    used_categories.add(category_name)

    return len(used_categories) * 15


# Штраф за хаотичность маршрута
def route_smoothness_penalty(route, df):

    if len(route) < 3:
        return 0

    penalty = 0

    prev_direction = None

    points = []

    points.append(startPoint)

    for idx in route:

        point = df.iloc[idx]

        points.append([
            float(point["lat"]),
            float(point["lon"])
        ])

    points.append(endPoint)

    for i in range(len(points) - 1):

        p1 = points[i]
        p2 = points[i + 1]

        direction = (
            p2[0] - p1[0],
            p2[1] - p1[1]
        )

        if prev_direction is not None:

            dot = (
                direction[0] * prev_direction[0]
                + direction[1] * prev_direction[1]
            )

            # Если движение назад
            if dot < 0:
                penalty += 50

        prev_direction = direction

    return penalty


# fitness-функция
def fitness_function(route, df):

    total_interest = 0

    for idx in route:

        point = df.iloc[idx]

        total_interest += point_interest_score(point)

    distance = route_distance(route, df)

    fitness = (
        total_interest * 20
        - distance * 0.03
    )

    return fitness

In [ ]:
POPULATION_SIZE = 100
GENERATIONS = 100

MIN_ROUTE_POINTS = 2
MAX_ROUTE_POINTS = 15

MUTATION_RATE = 0.2
TOURNAMENT_SIZE = 5

NUM_ISLANDS = 4

MIGRATION_INTERVAL = 20

MIGRATION_SIZE = 3

ISLAND_POPULATION_SIZE = POPULATION_SIZE // NUM_ISLANDS

# Расстояние между точками в метрах
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlambda / 2) ** 2
    )

    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))


# Длина маршрута
def route_distance(route, df):

    total_distance = 0

    prev_lat, prev_lon = startPoint

    for idx in route:
        point = df.iloc[idx]

        total_distance += haversine(
            prev_lat,
            prev_lon,
            point["lat"],
            point["lon"]
        )

        prev_lat = point["lat"]
        prev_lon = point["lon"]

    total_distance += haversine(
        prev_lat,
        prev_lon,
        endPoint[0],
        endPoint[1]
    )

    return total_distance


# Генерация случайной особи
def create_individual(df):

    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    route = random.sample(
        range(len(df)),
        route_size
    )

    return {
        "route": route,
        "fitness": None
    }


# Создание популяции
def create_population(df):
    return [
        create_individual(df)
        for _ in range(ISLAND_POPULATION_SIZE)
    ]


# Оценка популяции
def evaluate_population(population, df):

    for individual in population:
        individual["fitness"] = fitness_function(
            individual["route"],
            df
        )


# Турнир
def tournament_selection(population):

    tournament = random.sample(
        population,
        TOURNAMENT_SIZE
    )

    tournament.sort(
        key=lambda x: x["fitness"],
        reverse=True
    )

    return tournament[0]


# Кроссовер
def crossover(parent1, parent2):

    route1 = parent1["route"]
    route2 = parent2["route"]

    if len(route1) < 2 or len(route2) < 2:
        return {
            "route": route1.copy(),
            "fitness": None
        }

    cut1 = random.randint(1, len(route1) - 1)
    cut2 = random.randint(1, len(route2) - 1)

    child_route = route1[:cut1]

    for point in route2[cut2:]:
        if point not in child_route:
            child_route.append(point)

    return {
        "route": child_route,
        "fitness": None
    }


# Мутация
def mutate(individual, df):

    route = individual["route"][:]

    if random.random() > MUTATION_RATE:
        return individual

    mutation_type = random.choice([
        "swap",
        "remove",
        "add",
        "shuffle"
    ])

    if mutation_type == "swap" and len(route) >= 2:

        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]

    elif mutation_type == "remove":

        if len(route) > MIN_ROUTE_POINTS:
            idx = random.randint(0, len(route) - 1)
            route.pop(idx)

    elif mutation_type == "add":

        

        if len(route) < MAX_ROUTE_POINTS:

            available = list(
                set(range(len(df))) - set(route)
            )

            if available:

                available_slots = MAX_ROUTE_POINTS - len(route)

                count_to_add = min(
                    random.randint(1, 3),
                    available_slots
                )

                for _ in range(count_to_add):

                    if not available:
                        break

                    new_point = random.choice(available)
                    available.remove(new_point)

                    insert_pos = random.randint(
                        0,
                        len(route)
                    )

                    route.insert(insert_pos, new_point)

    elif mutation_type == "shuffle":

        random.shuffle(route)

    individual["route"] = route
    individual["fitness"] = None

    return individual


# Создание нового поколения
def create_next_generation(population, df):

    new_population = []

    population.sort(
        key=lambda x: x["fitness"],
        reverse=True
    )

    elite = population[:5]

    new_population.extend(elite)

    while len(new_population) < POPULATION_SIZE:

        parent1 = tournament_selection(population)
        parent2 = tournament_selection(population)

        child = crossover(parent1, parent2)

        child = mutate(child, df)
        
        child["fitness"] = fitness_function(child["route"], df)

        new_population.append(child)

    return new_population

# def safe_sort(island):
#     return sorted(
#         island,
#         key=lambda x: x["fitness"] if x["fitness"] is not None else -1,
#         reverse=True
#     )

def migrate(islands):

    migrants = []

    # собираем лучших
    for island in islands:

        for island in islands:
            for ind in island:
                if ind["fitness"] is None:
                    print("None fitness found")

        island.sort(key=lambda x: x["fitness"], reverse=True)

        migrants.append(
            copy.deepcopy(island[:MIGRATION_SIZE])
        )

    # кольцевая миграция
    for i in range(NUM_ISLANDS):

        next_island = (i + 1) % NUM_ISLANDS

        islands[next_island].sort(
            key=lambda x: x["fitness"]
        )

        islands[next_island] = (
            migrants[i]
            + islands[next_island][MIGRATION_SIZE:]
        )

    return islands

# Генетический алгоритм
def genetic_algorithm(df):

    islands = [
        create_population(df)
        for _ in range(NUM_ISLANDS)
    ]

    for generation in range(GENERATIONS):
        # print("Поколение:", generation)
        for island_idx in range(NUM_ISLANDS):
            # print("\остров:", island_idx)

            evaluate_population(
                islands[island_idx],
                df
            )

            islands[island_idx] = create_next_generation(
                islands[island_idx],
                df
            )
            

        if generation > 0 and generation % MIGRATION_INTERVAL == 0:
            islands = migrate(islands)

    all_individuals = []

    for island in islands:

        evaluate_population(island, df)

        all_individuals.extend(island)

    best = max(
        all_individuals,
        key=lambda x: x["fitness"]
    )

    return best


# Полный маршрут
def project_point_to_line(point, start, end):
    """
    Проекция точки на линию START -> END.
    Возвращает параметр t:
    0   = начало линии
    1   = конец линии
    >1  = дальше конца
    <0  = до начала
    """

    px, py = point
    sx, sy = start
    ex, ey = end

    line_vec = np.array([ex - sx, ey - sy])
    point_vec = np.array([px - sx, py - sy])

    line_len_sq = np.dot(line_vec, line_vec)

    if line_len_sq == 0:
        return 0

    t = np.dot(point_vec, line_vec) / line_len_sq

    return t


def build_full_route(best_individual, df):

    route_points = []

    points = []

    for idx in best_individual["route"]:

        point = df.iloc[idx]

        lat = float(point["lat"])
        lon = float(point["lon"])

        t = project_point_to_line(
            [lat, lon],
            startPoint,
            endPoint
        )

        points.append({
            "name": point["name"],
            "lat": lat,
            "lon": lon,
            "t": t
        })

    points.sort(key=lambda x: x["t"])

    route_points.append({
        "name": "START",
        "lat": startPoint[0],
        "lon": startPoint[1]
    })

    route_points.extend(points)

    route_points.append({
        "name": "END",
        "lat": endPoint[0],
        "lon": endPoint[1]
    })

    return route_points

# Ссылка на яндекс карты
def generate_yandex_maps_url(route_points):
    """
    route_points:
    [
        {"name": "...", "lat": ..., "lon": ...},
        ...
    ]
    """

    rtext = "~".join(
        f"{point['lat']},{point['lon']}"
        for point in route_points
    )

    yandex_url = (
        f"https://yandex.ru/maps/"
        f"?mode=routes"
        f"&rtext={rtext}"
        f"&rtt=pd"
    )

    return yandex_url

# Запуск

best_individual = genetic_algorithm(df)

final_route = build_full_route(
    best_individual,
    df
)

print("СПИСОК МЕСТ")

places_names = []

for point in final_route:

    # START / END можно исключить
    if point["name"] not in ["START", "END"]:
        places_names.append(point["name"])

for i, name in enumerate(places_names):

    print(f"{i + 1}. {name}")


yandex_url = generate_yandex_maps_url(
    final_route
)

print("ССЫЛКА НА МАРШРУТ")

print(yandex_url)

Поколение: 0
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 1
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 2
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 3
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 4
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 5
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 6
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 7
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 8
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 9
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 10
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 11
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 12
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 13
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 14
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 15
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 16
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 17
\остров: 0

In [26]:
def route_interest_score(route, df):
    total = 0

    for idx in route:
        point = df.iloc[idx]
        total += point_interest_score(point)

    return total

def run_experiment(df, user_prefs, experiment_name="exp"):

    start_time = time.time()

    best_individual = genetic_algorithm(df)

    final_route = build_full_route(best_individual, df)

    end_time = time.time()
    
    interest = route_interest_score(best_individual["route"], df)
    distance = route_distance(best_individual["route"], df)
    duration = end_time - start_time

    yandex_url = generate_yandex_maps_url(final_route)

    route_text = route_to_names(final_route)

    return {
        "experiment": experiment_name,
        "interest_score": interest,
        "distance": distance,
        "time_sec": duration,
        "route_length": len(best_individual["route"]),
        "route_text": route_text,
        "yandex_url": yandex_url
    }


def run_multiple_experiments(df, experiments):

    results = []

    for exp in experiments:

        print(f"\nRunning: {exp['name']}")

        result = run_experiment(
            df=df,
            user_prefs=exp["user_prefs"],
            experiment_name=exp["name"]
        )

        results.append(result)

        print(f"Done: {exp['name']} | time: {result['time_sec']:.2f}s")

    return results

def route_to_names(final_route):

    names = []

    for point in final_route:

        if point["name"] not in ["START", "END"]:
            names.append(point["name"])

    return " -> ".join(names)

def save_results(results, filename="island_experiments.csv"):

    rows = []

    for r in results:

        rows.append({
            "experiment": r["experiment"],
            "interest_score": r["interest_score"],
            "distance": r["distance"],
            "time_sec": r["time_sec"],
            "route_length": r["route_length"],
            "route_text": r["route_text"],
            "yandex_url": r["yandex_url"]
        })

    df_results = pd.DataFrame(rows)
    df_results.to_csv(filename, index=False)

    print(f"\nSaved to {filename}")

results = run_multiple_experiments(df, experiments)

save_results(results)


Running: user_1_military_focus
Поколение: 0
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 1
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 2
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 3
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 4
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 5
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 6
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 7
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 8
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 9
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 10
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 11
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 12
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 13
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 14
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 15
\остров: 0
\остров: 1
\остров: 2
\остров: 3
Поколение: 16
\остров: 0
\остров: 1
\остров: 2
\ос